<a href="https://colab.research.google.com/github/Vishu235/bears/blob/main/colab/BEARS_Colab_Run.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BEARS Colab Pro Runbook

Run this notebook from top to bottom. You should only need to edit the first config cell if your GitHub repo or Google Drive paths differ.

What this notebook includes:

- Colab-safe dependency setup that keeps Colab's CUDA PyTorch instead of downgrading to the original 2022 pins.
- Path handling for Google Drive folders with spaces, such as `PES - Semester 4`.
- PyTorch 2.6+ compatibility for old dataset artifacts saved with NumPy arrays.
- Smoke tests for HalfMNIST training, optional HalfMNIST checkpoint evaluation, optional MiniKandinsky, and BDD-OIA preprocessing/training.
- Optional full practical BDD-OIA run using ResNet50 ImageNet features from `lastframe.zip`.
- Result archiving and copy-back to Google Drive.

Note: the BDD full run here uses ResNet50 replacement features. It is useful for running the pipeline, but it is not the exact paper-faithful Faster-RCNN/CBM `bdd_2048.zip` feature setup.

In [ ]:
#@title 1. Configuration
# Fresh-runtime defaults for reproducing the HalfMNIST BEARS paper row.

REPO_URL = "https://github.com/Vishu235/bears.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
REPO_DIR = "/content/bears"  #@param {type:"string"}

DRIVE_DATA_DIR = "/content/drive/MyDrive/PES - Semester 4/bears_data"  #@param {type:"string"}
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/PES - Semester 4/bears_results"  #@param {type:"string"}

LASTFRAME_ZIP = f"{DRIVE_DATA_DIR}/lastframe.zip"
HALFMNIST_CKPT_NAME = "halfmnist-mnistdpl-dis-None-end.pt"
HALFMNIST_CKPT_IN_DRIVE = f"{DRIVE_DATA_DIR}/{HALFMNIST_CKPT_NAME}"

# HalfMNIST reproduction switches.
RUN_HALFMNIST_SMOKE = False  #@param {type:"boolean"}
RUN_HALFMNIST_BASELINE_EVALS = True  #@param {type:"boolean"}
RUN_HALFMNIST_BASELINE_OOD_EVALS = True  #@param {type:"boolean"}
RUN_HALFMNIST_PAPER_BEARS = True  #@param {type:"boolean"}
RUN_HALFMNIST_PAPER_BEARS_OOD = True  #@param {type:"boolean"}

# Restore/copy previous practical BDD outputs if they exist in Drive.
RESTORE_PREVIOUS_BDD_OUTPUTS = True  #@param {type:"boolean"}

# Optional datasets. Keep these off for the focused HalfMNIST rerun.
RUN_MINIKAND_SMOKE = False  #@param {type:"boolean"}
RUN_BDD_SMOKE = False  #@param {type:"boolean"}
RUN_FULL_BDD = False  #@param {type:"boolean"}

# Full BDD settings, used only when RUN_FULL_BDD is true.
FULL_BDD_EPOCHS = 30  #@param {type:"integer"}
FULL_BDD_BATCH_SIZE = 256  #@param {type:"integer"}
FULL_BDD_FEATURE_BATCH_SIZE = 64  #@param {type:"integer"}
FULL_BDD_W_ENTROPY = 1.0  #@param {type:"number"}

# BDD smoke settings.
BDD_SMOKE_LIMIT_PER_SPLIT = 8  #@param {type:"integer"}
BDD_SMOKE_FEATURE_BATCH_SIZE = 8  #@param {type:"integer"}

print("Configuration ready.")


In [ ]:
#@title 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title 3. Clone or update repository
import os
import subprocess
from pathlib import Path

repo_dir = Path(REPO_DIR)
if repo_dir.exists() and (repo_dir / ".git").exists():
    os.chdir(repo_dir)
    subprocess.run(["git", "fetch", "origin"], check=True)
    subprocess.run(["git", "checkout", BRANCH], check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], check=True)
elif repo_dir.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repo. Remove it or choose another REPO_DIR.")
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
    os.chdir(repo_dir)

print("Repo ready:", Path.cwd())
subprocess.run(["git", "log", "--oneline", "-3"], check=False)

In [ ]:
#@title 4. Install Colab-safe dependencies
import os
import subprocess

os.chdir(REPO_DIR)
subprocess.run(["python", "-m", "pip", "install", "-q", "--upgrade", "pip"], check=True)
subprocess.run(["python", "-m", "pip", "install", "-q", "-r", "requirements.colab.txt"], check=True)
print("Dependencies installed.")

In [ ]:
#@title 5. Diagnostics
import os
import subprocess

os.chdir(REPO_DIR)
subprocess.run(["python", "colab_runner.py", "--job", "diagnostics"], check=True)

In [ ]:
#@title 6. Check data files and restore optional checkpoint
import shutil
from pathlib import Path

repo_dir = Path(REPO_DIR)
drive_data_dir = Path(DRIVE_DATA_DIR)
drive_results_dir = Path(DRIVE_RESULTS_DIR)
lastframe_zip = Path(LASTFRAME_ZIP)
repo_ckpt = repo_dir / "XOR_MNIST" / "data" / "ckpts" / HALFMNIST_CKPT_NAME

checkpoint_candidates = [
    Path(HALFMNIST_CKPT_IN_DRIVE),
    drive_results_dir / "halfmnist_ckpts" / HALFMNIST_CKPT_NAME,
    drive_results_dir / HALFMNIST_CKPT_NAME,
    Path("/content") / HALFMNIST_CKPT_NAME,
    repo_ckpt,
]

print("Drive data dir:", drive_data_dir, "exists=", drive_data_dir.exists())
print("Drive results dir:", drive_results_dir, "exists=", drive_results_dir.exists())
print("lastframe.zip:", lastframe_zip, "exists=", lastframe_zip.exists())

repo_ckpt.parent.mkdir(parents=True, exist_ok=True)
for candidate in checkpoint_candidates:
    print("Checkpoint candidate:", candidate, "exists=", candidate.exists())

source_ckpt = next((p for p in checkpoint_candidates if p.exists()), None)
if source_ckpt is not None and source_ckpt.resolve() != repo_ckpt.resolve():
    shutil.copy2(source_ckpt, repo_ckpt)
    print("Copied HalfMNIST checkpoint from", source_ckpt, "to", repo_ckpt)
elif repo_ckpt.exists():
    print("HalfMNIST checkpoint is ready:", repo_ckpt)
else:
    print("HalfMNIST checkpoint not found. Upload/copy it before running paper BEARS eval.")

kand_zip = repo_dir / "XOR_MNIST" / "data" / "kand-3k.zip"
print("MiniKandinsky zip:", kand_zip, "exists=", kand_zip.exists())


In [ ]:
#@title 7. Helper used by all smoke/full jobs
import os
import shlex
import subprocess
from pathlib import Path

def run_job(job, *, required=True, extra_args=None):
    args = [
        "python", "colab_runner.py",
        "--job", job,
        "--lastframe-zip", LASTFRAME_ZIP,
    ]
    if extra_args:
        args.extend(str(item) for item in extra_args)

    print("\n>>> Running:", " ".join(shlex.quote(arg) for arg in args), flush=True)
    completed = subprocess.run(args, cwd=REPO_DIR, check=False)
    if completed.returncode:
        logs = sorted((Path(REPO_DIR) / "logs").glob("*.log"), key=lambda p: p.stat().st_mtime)
        if logs:
            latest = logs[-1]
            print(f"\n--- Tail of latest log: {latest} ---")
            print("".join(latest.read_text(errors="replace").splitlines(True)[-120:]))
        message = f"Job {job} failed with exit code {completed.returncode}. See logs/ for full output."
        if required:
            raise SystemExit(message)
        print("SKIPPED/FAILED:", message)
    else:
        print(f"<<< Completed {job}")
    return completed.returncode

print("Helper ready.")

## HalfMNIST Paper Reproduction
Run these cells in a fresh Colab runtime after setup. The BEARS cells use the paper-style preset added to `colab_runner.py`.


In [ ]:
#@title 8. HalfMNIST smoke and baseline evals
from pathlib import Path

repo_ckpt = Path(REPO_DIR) / "XOR_MNIST" / "data" / "ckpts" / HALFMNIST_CKPT_NAME

if RUN_HALFMNIST_SMOKE:
    run_job("halfmnist_smoke", extra_args=["--epochs", "1", "--batch-size", "64"])

if RUN_HALFMNIST_BASELINE_EVALS:
    if not repo_ckpt.exists():
        raise FileNotFoundError(f"Missing HalfMNIST checkpoint: {repo_ckpt}")
    run_job("halfmnist_eval", extra_args=["--eval-type", "frequentist", "--seed", "0"])
    run_job("halfmnist_eval", extra_args=["--eval-type", "mcdropout", "--seed", "0"])
    if RUN_HALFMNIST_BASELINE_OOD_EVALS:
        run_job("halfmnist_eval", extra_args=["--eval-type", "frequentist", "--seed", "0", "--use-ood"])
        run_job("halfmnist_eval", extra_args=["--eval-type", "mcdropout", "--seed", "0", "--use-ood"])
else:
    print("Skipping HalfMNIST baseline evals.")


In [ ]:
#@title 9. Paper-style HalfMNIST BEARS reproduction
from pathlib import Path

repo_ckpt = Path(REPO_DIR) / "XOR_MNIST" / "data" / "ckpts" / HALFMNIST_CKPT_NAME
if not repo_ckpt.exists():
    raise FileNotFoundError(f"Missing HalfMNIST checkpoint: {repo_ckpt}")

paper_args = ["--eval-type", "bears", "--halfmnist-preset", "paper"]
if RUN_HALFMNIST_PAPER_BEARS:
    run_job("halfmnist_eval", extra_args=paper_args)
else:
    print("Skipping in-distribution paper-style BEARS eval.")

if RUN_HALFMNIST_PAPER_BEARS_OOD:
    run_job("halfmnist_eval", extra_args=paper_args + ["--use-ood"])
else:
    print("Skipping OOD paper-style BEARS eval.")


## Optional Restores And Extra Datasets
These are useful if you want BDD practical summaries or quick smoke tests, but they are not required for the HalfMNIST BEARS rerun.


In [ ]:
#@title 10. Restore previous BDD outputs and summaries from Drive
import shutil
from pathlib import Path

if RESTORE_PREVIOUS_BDD_OUTPUTS:
    drive_results_dir = Path(DRIVE_RESULTS_DIR)
    restore_pairs = [
        (
            drive_results_dir / "bdd_out_dpl_auc_entropy-42",
            Path(REPO_DIR) / "BDD_OIA" / "out" / "bdd" / "dpl_auc_entropy-42",
        ),
        (
            drive_results_dir / "bdd_model_dpl_auc_entropy-42",
            Path(REPO_DIR) / "BDD_OIA" / "models" / "bdd" / "dpl_auc_entropy-42",
        ),
        (
            drive_results_dir / "summary_tables",
            Path(REPO_DIR) / "summary_tables",
        ),
    ]
    for src, dst in restore_pairs:
        if src.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copytree(src, dst, dirs_exist_ok=True)
            print("Restored", src, "->", dst)
        else:
            print("Not found in Drive, skipping:", src)
else:
    print("Skipping restore of previous BDD outputs.")


In [ ]:
#@title 11. Optional MiniKandinsky smoke
if RUN_MINIKAND_SMOKE:
    run_job("minikand_smoke", extra_args=["--epochs", "1", "--batch-size", "16"])
else:
    print("Skipping MiniKandinsky smoke.")


In [ ]:
#@title 12. Optional BDD-OIA smoke
from pathlib import Path

if RUN_BDD_SMOKE:
    if not Path(LASTFRAME_ZIP).exists():
        print("Skipping BDD smoke because lastframe.zip is missing:", LASTFRAME_ZIP)
    else:
        run_job(
            "bdd_preprocess_smoke",
            extra_args=[
                "--feature-weights", "none",
                "--feature-batch-size", str(BDD_SMOKE_FEATURE_BATCH_SIZE),
                "--limit-per-split", str(BDD_SMOKE_LIMIT_PER_SPLIT),
            ],
        )
        run_job("bdd_train_smoke")
else:
    print("Skipping BDD smoke.")


In [ ]:
#@title 13. Optional full practical BDD run
from pathlib import Path

if RUN_FULL_BDD:
    if not Path(LASTFRAME_ZIP).exists():
        raise FileNotFoundError(f"Missing lastframe.zip: {LASTFRAME_ZIP}")
    run_job(
        "bdd_preprocess_full",
        extra_args=[
            "--feature-weights", "imagenet",
            "--feature-batch-size", str(FULL_BDD_FEATURE_BATCH_SIZE),
        ],
    )
    run_job(
        "bdd_train_full",
        extra_args=[
            "--epochs", str(FULL_BDD_EPOCHS),
            "--bdd-batch-size", str(FULL_BDD_BATCH_SIZE),
            "--w-entropy", str(FULL_BDD_W_ENTROPY),
        ],
    )
else:
    print("Skipping full practical BDD run.")


## Summaries And Artifacts
Generate CSV tables, archive the run, and copy the files back to Drive.


In [ ]:
#@title 14. Generate report summary tables
import csv
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

repo_dir = Path(REPO_DIR)
out_dir = repo_dir / "summary_tables"
out_dir.mkdir(exist_ok=True)

def scalar(value):
    if isinstance(value, list):
        return scalar(value[0])
    return float(value)

def halfmnist_priority(row):
    name = row["Source file"]
    if row["Method"] == "DPL + BEARS":
        if "seed_0" in name and "lambda_0.8" in name and "real-kl_True" in name:
            return 0
        return 10
    if "seed_0" in name:
        return 1
    return 5

method_map = {
    "frequentist": "DPL / frequentist",
    "mcdropout": "DPL + MCDO",
    "bears": "DPL + BEARS",
}

half_rows = []
for path in sorted((repo_dir / "XOR_MNIST" / "dumps").glob("*halfmnist*.json")):
    name = path.name
    split = "OOD" if "ood_True" in name else "ID"
    method_key = next((key for key in method_map if f"incomplete_{key}" in name), None)
    if method_key is None:
        continue
    data = json.loads(path.read_text())
    half_rows.append({
        "Dataset": "MNIST-Half",
        "Split": split,
        "Method": method_map[method_key],
        "AccY": scalar(data.get("yac", np.nan)),
        "AccC": scalar(data.get("cac", np.nan)),
        "F1Y": scalar(data.get("yf1", np.nan)),
        "F1C": scalar(data.get("cf1", np.nan)),
        "ECEY": scalar(data.get("ece y", np.nan)),
        "ECEC": scalar(data.get("ece", np.nan)),
        "Source file": name,
    })

half_all = pd.DataFrame(half_rows)
if not half_all.empty:
    half_all["Priority"] = half_all.apply(halfmnist_priority, axis=1)
    half_preferred = (
        half_all.sort_values(["Split", "Method", "Priority", "Source file"])
        .groupby(["Split", "Method"], as_index=False)
        .first()
        .drop(columns=["Priority"])
    )
    half_all.drop(columns=["Priority"]).to_csv(out_dir / "halfmnist_all_runs.csv", index=False)
    half_preferred.to_csv(out_dir / "halfmnist_summary.csv", index=False)
    display(half_preferred)
else:
    print("No HalfMNIST JSON dumps found yet.")


def binary_ece(y_true, prob, n_bins=10):
    y_true = np.asarray(y_true).astype(int)
    prob = np.clip(np.asarray(prob).astype(float), 0.0, 1.0)
    y_pred = (prob >= 0.5).astype(int)
    conf = np.where(y_pred == 1, prob, 1.0 - prob)
    correct = (y_pred == y_true).astype(float)
    ece = 0.0
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (conf >= lo) & (conf <= hi) if i == 0 else (conf > lo) & (conf <= hi)
        if mask.any():
            ece += mask.mean() * abs(correct[mask].mean() - conf[mask].mean())
    return float(ece)

def mean_binary_ece(y_true, prob, cols=None):
    if cols is None:
        cols = range(y_true.shape[1])
    return float(np.mean([binary_ece(y_true[:, i], prob[:, i]) for i in cols]))

bdd_path = repo_dir / "BDD_OIA" / "out" / "bdd" / "dpl_auc_entropy-42" / "test_results_of_BDD.csv"
if bdd_path.exists() and bdd_path.stat().st_size > 0:
    rows = []
    with bdd_path.open() as handle:
        for row in csv.reader(handle):
            vals = [float(x) for x in row if x != ""]
            if vals:
                rows.append(vals)
    arr = np.asarray(rows, dtype=float)
    y_true_5 = arr[:, 0:5]
    out_8 = arr[:, 5:13]
    c_prob = arr[:, 13:34]
    c_true = arr[:, -21:]
    y_true_4 = y_true_5[:, 0:4]
    y_prob_4 = out_8[:, [1, 3, 5, 7]]
    y_pred_4 = (y_prob_4 >= 0.5).astype(int)
    c_pred = (c_prob >= 0.5).astype(int)
    fs_cols = list(range(0, 9))
    left_cols = list(range(9, 15))
    right_cols = list(range(15, 21))
    bdd_df = pd.DataFrame([{
        "Dataset": "BDD-OIA practical reconstruction",
        "Method": "DPL-AUC + entropy",
        "Feature source": "ResNet50 ImageNet 2048 features from lastframe.zip",
        "mF1(Y) 4 actions": f1_score(y_true_4, y_pred_4, average="macro", zero_division=0),
        "mF1(C)": f1_score(c_true, c_pred, average="macro", zero_division=0),
        "mECEY 4 actions": mean_binary_ece(y_true_4, y_prob_4),
        "mECEC": mean_binary_ece(c_true, c_prob),
        "ECEC(F,S)": mean_binary_ece(c_true, c_prob, fs_cols),
        "ECEC(L)": mean_binary_ece(c_true, c_prob, left_cols),
        "ECEC(R)": mean_binary_ece(c_true, c_prob, right_cols),
        "Action exact-match acc": accuracy_score(y_true_4, y_pred_4),
        "Concept exact-match acc": accuracy_score(c_true, c_pred),
        "Source file": str(bdd_path),
    }])
    bdd_df.to_csv(out_dir / "bdd_practical_summary.csv", index=False)
    display(bdd_df)
else:
    print("No non-empty BDD test CSV found; skipping BDD practical summary:", bdd_path)

print("Summary files:")
for path in sorted(out_dir.glob("*.csv")):
    print(path)


In [ ]:
#@title 15. Archive results
run_job("archive_results")

from pathlib import Path
for path in sorted((Path(REPO_DIR) / "colab_outputs").glob("bears_results_*.zip")):
    print(path, path.stat().st_size, "bytes")


In [ ]:
#@title 16. Copy archive and summaries to Drive
import shutil
from pathlib import Path

result_dir = Path(DRIVE_RESULTS_DIR)
result_dir.mkdir(parents=True, exist_ok=True)
archives = sorted((Path(REPO_DIR) / "colab_outputs").glob("bears_results_*.zip"), key=lambda p: p.stat().st_mtime)
if not archives:
    raise FileNotFoundError("No result archive found in colab_outputs.")
latest = archives[-1]
target = result_dir / latest.name
shutil.copy2(latest, target)
print("Copied", latest, "to", target)

summary_src = Path(REPO_DIR) / "summary_tables"
summary_dst = result_dir / "summary_tables"
if summary_src.exists():
    shutil.copytree(summary_src, summary_dst, dirs_exist_ok=True)
    print("Copied summaries to", summary_dst)
else:
    print("No summary_tables folder found to copy.")


## Optional Local Download
Use these only if Drive is hard to inspect. Change your browser download location away from `C:` first if local disk space is low.


In [ ]:
#@title 17. Optional local download zip without huge feature tensors
import subprocess
from pathlib import Path

zip_items = [
    "bears/summary_tables",
    "bears/XOR_MNIST/dumps",
    "bears/BDD_OIA/out",
    "bears/BDD_OIA/models/bdd/dpl_auc_entropy-42",
    "bears/logs",
    "bears/colab",
    "bears/colab_runner.py",
    "bears/requirements.colab.txt",
    "bears/README.md",
]
existing = [item for item in zip_items if (Path("/content") / item).exists()]
print("Adding to zip:")
for item in existing:
    print(" -", item)
if not existing:
    raise FileNotFoundError("No result paths found to zip.")
subprocess.run(["zip", "-r", "bears_results_only.zip", *existing], cwd="/content", check=True)
subprocess.run(["ls", "-lh", "/content/bears_results_only.zip"], check=True)


In [ ]:
#@title 18. Optional trigger browser download
from google.colab import files
files.download("/content/bears_results_only.zip")
